# exp03 — 보완재 엣지 Ablation

| 항목 | 값 |
|---|---|
| config | `experiments/configs/exp03_complement_edges.yaml` |
| 결과 저장 | `experiments/results/exp03_complement_edges/` |
| 변경점 | `include_complement_edges` false → **true** (968쌍, Lift≥3.0) |
| 기반 세팅 | exp02 (α튜닝) 위에 보완재 엣지만 추가 |
| 목적 | `product-complement-product` 동반구매 관계가 NPD 성공 예측·추천에 기여하는지 ablation 검증 |
| 가설 | 동반구매 맥락이 제품 임베딩을 풍부하게 해 PR-AUC ↑, 추천 다양성 ↑ |
| 주의 | complement 엣지는 세븐일레븐 POS 기반(ITEM_CD). CU/GS25 제품 노드에는 미적용됨. |

In [ ]:
import os, sys
ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)

import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

from experiments.exp_utils import (
    run_experiment, load_experiment, compare_experiments,
    plot_alpha_heatmap, plot_training_curve, print_metrics_table, print_recommendations
)

EXP_NAME = 'exp03_complement_edges'
CFG_PATH = 'experiments/configs/exp03_complement_edges.yaml'
print('ROOT:', ROOT)

## 0. 보완재 데이터 확인

In [ ]:
import pandas as pd
cp = pd.read_csv('data/processed/complement_lift_pairs.csv')
print(f'보완재 쌍: {len(cp):,}행')
print(f'Lift 분포:\n{cp["향상도(Lift)"].describe().round(1)}')
display(cp.head(5))

## 1. 학습 실행

In [ ]:
results = run_experiment(CFG_PATH, EXP_NAME)

## 2. 성능 지표

In [ ]:
print_metrics_table(results)

## 3. 학습 곡선

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 4))
plot_training_curve(results.get('history', []), ax=ax)
plt.savefig(f'experiments/results/{EXP_NAME}/training_curve.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. DiffMG α_r — complement 엣지 포함 후 관계 중요도 변화

> `product__complement__product` 관계가 α_r 에서 높은 가중치를 받으면
> 동반구매 관계가 NPD 성공 예측에 실질적으로 기여함.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 3))
try:
    plot_alpha_heatmap('exp02_alpha_tuning', ax=axes[0])
    axes[0].set_title('exp02 (α tuned, complement=False)')
except Exception as e:
    axes[0].set_title(f'exp02 로드 실패: {e}')
plot_alpha_heatmap(EXP_NAME, ax=axes[1])
axes[1].set_title('exp03 (complement 엣지 추가)')
plt.savefig(f'experiments/results/{EXP_NAME}/alpha_heatmap_compare.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. 순회 추천 — 보완재 엣지 반영 후 변화

> 동반구매 컨텍스트가 추천 결과에 반영되면 추천 키워드 분포가 달라질 수 있음.

In [ ]:
print('=== exp03 추천 ===')
print_recommendations(results)

try:
    r02 = load_experiment('exp02_alpha_tuning')
    print('\n=== exp02 추천 (비교) ===')
    print_recommendations(r02)
except FileNotFoundError:
    pass

## 6. 전체 실험 비교

In [ ]:
df = compare_experiments(['exp01_baseline', 'exp02_alpha_tuning', 'exp03_complement_edges'])
display(df[['exp', 'val_pr_auc', 'val_auc_roc', 'test_pr_auc', 'test_auc_roc', 'test_f1']])

## 7. Ablation 결론 메모

아래 셀에 결과 해석 및 결론을 기록하세요.

### complement 엣지 효과 판단

- [ ] PR-AUC 상승 → 보완재 관계가 유의미. v2 학습에 포함 결정.
- [ ] PR-AUC 하락 또는 유사 → 채널 비대칭(세븐만 존재) 부작용. 제외 유지.
- [ ] α_r에서 `complement` 관계 가중치: __(기록)__

### exp02 vs exp03 test PR-AUC 차이
- exp02: __(기록)__
- exp03: __(기록)__
- Δ: __(기록)__

### 다음 단계
- complement 효과 있음 → Lift 임계값 조정 실험
- complement 효과 없음 → 대체재(substitute) 엣지 또는 영수증 노드 도입 검토